# Treinamento numérico de uma rede neural 2–8–1

Este notebook reproduz os valores utilizados na animação. O conjunto de dados, os snapshots, as ativações, a perda e os gradientes são importados do módulo de treinamento do repositório.

[![Fronteira de decisão final](../assets/images/04-final-boundary.png)](../assets/videos/neural-network-learning-1080x1350-2x.mp4)

[Abrir o vídeo final em 1080 × 1350](../assets/videos/neural-network-learning-1080x1350-2x.mp4)

In [ ]:
from pathlib import Path
import sys
import numpy as np

ROOT = Path.cwd().resolve()
if not (ROOT / "neural_training_numpy.py").is_file():
    ROOT = ROOT.parent
assert (ROOT / "neural_training_numpy.py").is_file(), "Execute a partir da raiz ou de notebooks/."
sys.path.insert(0, str(ROOT))

from neural_training_numpy import (
    forward,
    gradient_at_snapshot,
    hidden_state,
    predict,
    sample_binary_cross_entropy,
    train_network,
)
from validate_neural_training import best_linear_accuracy

np.set_printoptions(precision=6, suppress=True)

## 1. Dados e configuração

São geradas duas meias-luas entrelaçadas, com 14 observações por classe. A semente 13 fixa tanto o ruído dos dados quanto a inicialização dos parâmetros. A rede usa duas entradas, oito neurônios ReLU e uma saída sigmoide.

In [ ]:
bundle = train_network()
print("shape de X:", bundle.x.shape)
print("contagem por classe:", np.bincount(bundle.y.astype(int)))
print("taxa de aprendizado:", bundle.learning_rate)
print("snapshots:", sorted(bundle.snapshots))
print("amostra selecionada:", bundle.selected_sample)
print("neurônio oculto selecionado:", bundle.selected_hidden_unit + 1)

### Não separabilidade linear

Uma busca angular avalia limiares lineares nas duas orientações de classe. O melhor resultado permanece abaixo de 100%, justificando a necessidade de uma fronteira não linear.

In [ ]:
linear_accuracy = best_linear_accuracy(bundle.x, bundle.y)
print(f"melhor acurácia linear: {linear_accuracy:.6f}")
assert linear_accuracy < 1.0

## 2. Normalização

Cada variável é centralizada e dividida pelo desvio-padrão calculado sobre as 28 observações:

$$
\\tilde{\\mathbf{x}}=\\frac{\\mathbf{x}-\\boldsymbol{\\mu}}{\\boldsymbol{\\sigma}}.
$$

In [ ]:
sample_index = bundle.selected_sample
sample = bundle.x[[sample_index]]
sample_normalized = (sample - bundle.mean) / bundle.scale

print("observação original:", sample[0])
print("média:", bundle.mean)
print("escala:", bundle.scale)
print("observação normalizada:", sample_normalized[0])

## 3. Combinação afim e ReLU

A camada oculta calcula

$$
\\mathbf{z}^{(1)}=\\tilde{\\mathbf{X}}\\mathbf{W}^{(1)}+\\mathbf{b}^{(1)},
\\qquad
\\mathbf{h}=\\max(0,\\mathbf{z}^{(1)}).
$$

A imagem mostra a substituição termo a termo usada no vídeo.

![Substituição numérica](../assets/images/01-real-arithmetic.png)

In [ ]:
snapshot_0 = bundle.snapshots[0]
z1, hidden, output_logit, probability = forward(
    sample_normalized,
    snapshot_0.w1,
    snapshot_0.b1,
    snapshot_0.w2,
    snapshot_0.b2,
)
unit = bundle.selected_hidden_unit

print("pesos do neurônio selecionado:", snapshot_0.w1[:, unit])
print("viés do neurônio selecionado:", snapshot_0.b1[unit])
print(f"z_{unit + 1}: {z1[0, unit]:.9f}")
print(f"h_{unit + 1}: {hidden[0, unit]:.9f}")
assert np.isclose(hidden[0, unit], max(0.0, z1[0, unit]))

### Interpretação da ReLU

$$
\\operatorname{ReLU}(z)=\\max(0,z).
$$

Entradas negativas são anuladas; entradas positivas são preservadas. Para o neurônio selecionado, $z_2=0{,}687$ e $h_2=0{,}687$.

![Ativação ReLU](../assets/images/02-relu-activation.png)

In [ ]:
z_check, h_check = hidden_state(bundle, snapshot_0, sample)
assert np.allclose(z_check, z1)
assert np.allclose(h_check, hidden)
print("ativações ocultas:", hidden[0])

## 4. Neurônio de saída

As oito ativações são combinadas por

$$
s=\\sum_{j=1}^{8}v_jh_j+c,
\\qquad
\\hat y=\\sigma(s).
$$

![Combinação das ativações](../assets/images/03-output-aggregation.png)

In [ ]:
target = float(bundle.y[sample_index])
sample_probability = float(probability[0])
sample_loss = sample_binary_cross_entropy(target, sample_probability)

print(f"logit de saída: {output_logit[0]:.9f}")
print(f"probabilidade: {sample_probability:.9f}")
print(f"classe correta: {target:.0f}")
print(f"BCE individual: {sample_loss:.9f}")

## 5. Evolução da perda

A entropia cruzada binária média é avaliada sobre as 28 observações. A tabela é obtida diretamente dos snapshots armazenados durante o gradiente descendente em lote completo.

In [ ]:
print("época | perda média | acurácia de treinamento")
for epoch in (0, 1, 10, 50, 100):
    snapshot = bundle.snapshots[epoch]
    print(f"{epoch:5d} | {snapshot.loss:11.6f} | {snapshot.accuracy:21.6f}")

assert np.all(np.isfinite(bundle.loss_history))
assert bundle.snapshots[100].loss < bundle.snapshots[0].loss
assert bundle.snapshots[100].accuracy == 1.0

## 6. Retropropagação

Os gradientes são derivados manualmente. A atualização de cada parâmetro segue

$$
w\\leftarrow w-\\eta\\frac{\\partial\\bar L}{\\partial w}.
$$

![Retropropagação e atualização de peso](../assets/images/05-backpropagation.png)

In [ ]:
dw1, db1, dw2, db2 = gradient_at_snapshot(bundle, snapshot_0)
weight_index = np.unravel_index(np.argmax(np.abs(dw1)), dw1.shape)
old_weight = snapshot_0.w1[weight_index]
gradient = dw1[weight_index]
calculated_weight = old_weight - bundle.learning_rate * gradient
observed_weight = bundle.snapshots[1].w1[weight_index]

print("índice do peso:", weight_index)
print(f"{old_weight:.9f} - {bundle.learning_rate:.2f} * {gradient:.9f} = {calculated_weight:.9f}")
print(f"valor armazenado na época 1: {observed_weight:.9f}")
assert np.isclose(calculated_weight, observed_weight, atol=1e-12)

## 7. Superfície de decisão

A fronteira branca corresponde ao nível $\\hat y(x_1,x_2)=0{,}5$ calculado com os oito neurônios ocultos. As quatro retas amarelas são níveis $z_j=0$ selecionados para legibilidade.

![Fronteira final](../assets/images/04-final-boundary.png)

In [ ]:
x1_values = np.linspace(0.0, 6.0, 121)
x2_values = np.linspace(0.0, 4.0, 81)
mesh_x1, mesh_x2 = np.meshgrid(x1_values, x2_values)
grid = np.column_stack((mesh_x1.ravel(), mesh_x2.ravel()))
surface = predict(bundle, bundle.snapshots[100], grid).reshape(mesh_x1.shape)

print("shape da superfície:", surface.shape)
print(f"probabilidade mínima: {surface.min():.6f}")
print(f"probabilidade máxima: {surface.max():.6f}")
assert surface.min() < 0.5 < surface.max()

## 8. Artefato audiovisual

O arquivo final possui 1080 × 1350, 60 FPS, 8.854 quadros e 147,567 segundos. A extensão temporal foi produzida pela duplicação de cada quadro, sem novas cenas ou interpolação.

<video controls width="540" src="../assets/videos/neural-network-learning-1080x1350-2x.mp4"></video>

Caso o visualizador não ofereça reprodução incorporada, utilize o [link direto para o MP4](../assets/videos/neural-network-learning-1080x1350-2x.mp4).

In [ ]:
video_path = ROOT / "assets" / "videos" / "neural-network-learning-1080x1350-2x.mp4"
image_paths = sorted((ROOT / "assets" / "images").glob("*.png"))

print("vídeo:", video_path.relative_to(ROOT), video_path.stat().st_size, "bytes")
print("imagens:", len(image_paths))
assert video_path.is_file()
assert len(image_paths) == 5

## Limitações

- O conjunto é sintético e contém somente 28 observações.
- A acurácia final é de treinamento e não mede generalização.
- A arquitetura 2–8–1 foi escolhida para visualização, sem busca de hiperparâmetros.
- As derivadas são implementadas manualmente, sem diferenciação automática.
- As fronteiras parciais não representam modelos treinados separadamente.